# 02 · LangChain: del modelo a la consulta

**Python que piensa: agentes de analítica con LangChain y LangGraph**
Sociedad Ecuatoriana de Estadística · Semana de Python

---

1. **Los datos**: qué tiene la tabla y cómo se lee un caso.
2. **El prompt**: el LLM escribe el SQL, sin diccionario y con diccionario.
3. **La caché**: el mismo diccionario, mucho más barato.
4. **La tool**: el LLM escribe el SQL y nuestro código lo ejecuta.
5. **La pregunta que rompe la cadena.**

> **Sobre los datos.** Reclamos médicos **sintéticos** del CMS de Estados
> Unidos. Sirven para construir herramientas, no para inferir sobre la población
> real de Medicare.

## Legibilidad para proyección

In [ ]:
from IPython.display import HTML, display

# Agranda la letra de las salidas para que se lea desde el fondo de la sala.
display(HTML("""
<style>
  .jp-RenderedText pre, .jp-OutputArea-output pre, div.output_area pre {
      font-size: 15px !important; line-height: 1.5 !important; }
  .jp-RenderedHTMLCommon table { font-size: 15px !important; }
  .jp-InputArea-editor, .CodeMirror { font-size: 15px !important; }
</style>
"""))
print("Tipografía ajustada para proyección.")

## Dónde estamos y credenciales

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# RAIZ = la carpeta del repositorio (la que contiene datos/).
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos").is_dir()), None)
BASE = RAIZ / "datos" / "reclamos.db"                      # la base SQLite
DICCIONARIO = RAIZ / "datos" / "diccionario_sabana.yaml"   # lo que el agente debe saber de la tabla

load_dotenv(RAIZ / ".env")   # las credenciales de los proveedores (ver notebook 01)

print(f"Base de datos : {BASE.name}  {'(existe)' if BASE.exists() else '(falta: corre 00_setup.ipynb)'}")
print(f"Diccionario   : {DICCIONARIO.name}")

## Los modelos

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

# El modelo de trabajo: gpt-4.1 en Azure AI Foundry.
foundry = ChatOpenAI(
    base_url=os.environ["AZURE_OPENAI_BASE_URL"],   # a dónde llamar
    api_key=os.environ["AZURE_OPENAI_API_KEY"],     # quién llama
    model=os.environ["AZURE_OPENAI_MODELO"],        # qué modelo
    temperature=0,                                  # para SQL: siempre la respuesta más probable
)

# Claude, para mostrar la caché con su marca explícita (parte 3).
claude = ChatAnthropic(model=os.environ["ANTHROPIC_MODELO"])

print("Foundry:", foundry.model_name, "| Claude:", claude.model)

## Lo que cuesta cada pregunta

Precios oficiales por millón de tokens, consultados el 2026-10-07:

| Modelo | Entrada | Lectura de caché | Escritura de caché | Salida | Fuente |
|---|---:|---:|---:|---:|---|
| gpt-4.1 (Foundry) | 2,00 | 0,50 | — (automática, sin recargo) | 8,00 | [OpenAI](https://developers.openai.com/api/docs/models/gpt-4.1), [Azure](https://azure.microsoft.com/es-es/pricing/details/azure-openai) |
| claude-sonnet-5-5 | 2,00 | 0,20 | 2,50 | 10,00 | [Anthropic](https://platform.claude.com/docs/en/about-claude/pricing) |

In [ ]:
# USD por MILLÓN de tokens. Si un proveedor cambia sus precios, se cambia aquí.
PRECIOS = {
    "gpt-4.1":           {"entrada": 2.00, "cache_lectura": 0.50, "cache_escritura": 2.00, "salida": 8.00},
    "claude-sonnet-5-5": {"entrada": 2.00, "cache_lectura": 0.20, "cache_escritura": 2.50, "salida": 10.00},
}
gasto_total = 0.0    # lo que llevamos gastado en este notebook, en USD

# Se usa después de CADA llamada: lee la cuenta de tokens y la multiplica por el precio.
def costo(respuesta, proveedor):
    global gasto_total
    modelo_usado = respuesta.response_metadata["model_name"]    # el modelo que respondió
    precio = next(p for nombre, p in PRECIOS.items() if modelo_usado.startswith(nombre))

    uso = respuesta.usage_metadata                              # la cuenta de tokens de esta respuesta
    detalle = uso["input_token_details"]
    leidos_de_cache = detalle.get("cache_read", 0)              # entrada que ya estaba en caché
    escritos_en_cache = (detalle.get("cache_creation", 0)       # entrada que se guardó en caché
                         or detalle.get("ephemeral_5m_input_tokens", 0))   # (Claude la informa aquí)
    entrada_normal = uso["input_tokens"] - leidos_de_cache - escritos_en_cache

    usd = (entrada_normal    * precio["entrada"]
         + leidos_de_cache   * precio["cache_lectura"]
         + escritos_en_cache * precio["cache_escritura"]
         + uso["output_tokens"] * precio["salida"]) / 1_000_000
    gasto_total += usd

    print(f"   [{proveedor} · {modelo_usado}] {uso['input_tokens']} tokens de entrada "
          f"({leidos_de_cache} de caché), {uso['output_tokens']} de salida "
          f"-> USD {usd:.6f} | acumulado USD {gasto_total:.6f}")
    return usd

---

# Parte 1 · Los datos

La tabla `sabana` tiene **una fila por reclamo**. Un reclamo es la cuenta que un
hospital o un médico le pasa a la aseguradora por una atención. Hay 656 147, de
2008 y 2009.

## Las variables con las que vamos a trabajar

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect(BASE)

pd.read_sql("""
    SELECT anio,                     -- año del reclamo: 2008 o 2009
           ambito,                   -- Hospitalario (internación) o Ambulatorio
           sexo, raza, banda_edad,   -- quién es el afiliado
           estado, codigo_condado,   -- dónde vive
           grupo_enfermedad,         -- el tipo de enfermedad del diagnóstico principal
           procedimiento_principal,  -- código del procedimiento (solo hospitalario)
           medico_tratante,          -- identificador del médico
           monto_pagado,             -- lo que pagó la aseguradora: el costo por defecto
           monto_total               -- lo que costó la atención, incluido lo que pagó el afiliado
    FROM sabana
    LIMIT 5
""", con)

## Cuántos valores distintos tiene cada una

In [ ]:
variables = ["anio", "ambito", "sexo", "raza", "banda_edad", "estado", "codigo_condado",
             "grupo_enfermedad", "procedimiento_principal", "medico_tratante"]

for variable in variables:
    distintos = con.execute(f"SELECT count(DISTINCT {variable}) FROM sabana").fetchone()[0]
    print(f"{variable:<24} {distintos:>7,} valores distintos")

## Un caso concreto

In [ ]:
# Los reclamos de 2009 de una sola afiliada.
caso = pd.read_sql("""
    SELECT fecha_inicio, ambito, grupo_enfermedad,
           monto_pagado,            -- lo que pagó la ASEGURADORA
           deducible, coaseguro,    -- lo que pagó la AFILIADA de su bolsillo
           monto_pagado_tercero,    -- lo que pagó otro seguro
           monto_total              -- la suma de los cuatro
    FROM sabana
    WHERE id_afiliado = '0011714C14B52EEB' AND anio = 2009
    ORDER BY fecha_inicio
""", con)
caso

In [ ]:
perfil = con.execute("""
    SELECT sexo, edad, estado FROM sabana
    WHERE id_afiliado = '0011714C14B52EEB' AND anio = 2009 LIMIT 1
""").fetchone()

print(f"Afiliada            : {perfil[0]}, {perfil[1]} años, vive en {perfil[2]}")
print(f"Reclamos en 2009    : {len(caso)} ({(caso['ambito'] == 'Hospitalario').sum()} hospitalario)")
print(f"Pagó la aseguradora : {caso['monto_pagado'].sum():>8,.2f}")
print(f"Costo de la atención: {caso['monto_total'].sum():>8,.2f}")
print()
print("Lo que vamos a preguntar suma esto mismo, para 79 692 personas.")

---

# Parte 2 · El prompt: el LLM escribe el SQL

El LLM no toca la base: **escribe un texto con un SQL**. Nosotros lo ejecutamos.
Lo que decide si ese SQL es correcto es **el contexto que le damos**.

## La plantilla del prompt

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Una plantilla es un prompt con huecos ({...}) que se llenan en cada pregunta.
plantilla = ChatPromptTemplate.from_messages([
    ("system", "Eres un analista de datos. Escribe UNA consulta SQL (SQLite) sobre la "
               "tabla sabana que responda la pregunta. Responde SOLO con el SQL.\n\n{contexto}"),
    ("human", "{pregunta}"),
])

# El operador | encadena piezas: lo que sale de la plantilla entra al modelo.
# Eso es una CADENA (chain): plantilla -> modelo.
cadena = plantilla | foundry

pregunta = "¿Cuánto se pagó en hospitalización en 2009?"
real = con.execute(
    "SELECT sum(monto_pagado) FROM sabana WHERE ambito = 'Hospitalario' AND anio = 2009"
).fetchone()[0]
print("Pregunta:", pregunta)
print(f"Respuesta correcta: {real:,.2f}")

## Escenario 1 · Sin diccionario: solo los nombres de las columnas

In [ ]:
columnas = [fila[1] for fila in con.execute("PRAGMA table_info(sabana)")]   # fila[1] = nombre
contexto_minimo = "Columnas de la tabla sabana: " + ", ".join(columnas)

respuesta = cadena.invoke({"contexto": contexto_minimo, "pregunta": pregunta})   # llena los huecos y pregunta

# El modelo a veces envuelve el SQL en ```sql ... ```: se lo quitamos.
sql = respuesta.text.strip().removeprefix("```sql").removesuffix("```").strip()

print("SQL que escribió:\n")
print(sql, "\n")
print("Resultado:", con.execute(sql).fetchall())
usd = costo(respuesta, "Foundry")

escenarios = []   # aquí guardamos cada escenario para compararlos al final
escenarios.append({"escenario": "1. Sin diccionario", "proveedor": "Foundry",
                   "tokens_entrada": respuesta.usage_metadata["input_tokens"],
                   "tokens_de_cache": respuesta.usage_metadata["input_token_details"].get("cache_read", 0),
                   "usd": usd})

## El diccionario de datos

In [ ]:
import yaml

# El diccionario dice qué significa cada columna, qué VALORES tiene, las trampas
# del dato y consultas de ejemplo con su resultado verificado.
diccionario = yaml.safe_load(DICCIONARIO.read_text(encoding="utf-8"))

# Los ejemplos se activan por módulo. En este notebook, solo los básicos.
diccionario["consultas_ejemplo"] = [e for e in diccionario["consultas_ejemplo"]
                                    if e["activa_desde"] <= "02"]

contexto = yaml.dump(diccionario, allow_unicode=True, sort_keys=False)   # de vuelta a texto

print(f"Columnas documentadas : {len(diccionario['columnas'])}")
print(f"Trampas del dato      : {len(diccionario['trampas'])}")
print(f"Consultas de ejemplo  : {len(diccionario['consultas_ejemplo'])}")
print(f"Largo del contexto    : {len(contexto):,} caracteres\n")
print("Así se ve una columna:\n")
print("ambito:", diccionario["columnas"]["ambito"])

## Escenario 2 · Con el diccionario en el prompt

In [ ]:
# (Si corres el notebook dos veces seguidas, aquí ya verás tokens de caché:
#  el proveedor guarda el prompt unos minutos.)
respuesta = cadena.invoke({"contexto": contexto, "pregunta": pregunta})
sql = respuesta.text.strip().removeprefix("```sql").removesuffix("```").strip()

print("SQL que escribió:\n")
print(sql, "\n")
print(f"Resultado: {con.execute(sql).fetchone()[0]:,.2f}   (correcto: {real:,.2f})")
usd = costo(respuesta, "Foundry")

escenarios.append({"escenario": "2. Con diccionario", "proveedor": "Foundry",
                   "tokens_entrada": respuesta.usage_metadata["input_tokens"],
                   "tokens_de_cache": respuesta.usage_metadata["input_token_details"].get("cache_read", 0),
                   "usd": usd})

---

# Parte 3 · La caché

El diccionario hace que el SQL salga bien, pero **cada pregunta lo vuelve a
mandar entero**, y se cobra cada vez. La **caché de prompts** guarda ese inicio
del prompt en el servidor del proveedor: si la siguiente llamada empieza
**exactamente igual**, esa parte se cobra mucho más barata.

## Escenario 3 · Foundry: la caché es automática

In [ ]:
# En Foundry (y en OpenAI) no hay que escribir nada: si el inicio del prompt
# es idéntico y largo (más de 1 024 tokens), el proveedor lo cachea solo.
# Otra pregunta, el MISMO contexto:
otra_pregunta = "¿Cuántos reclamos ambulatorios hubo en 2008?"

respuesta = cadena.invoke({"contexto": contexto, "pregunta": otra_pregunta})
print("SQL:", respuesta.text.strip())
usd = costo(respuesta, "Foundry")

escenarios.append({"escenario": "3. Con diccionario, en caché", "proveedor": "Foundry",
                   "tokens_entrada": respuesta.usage_metadata["input_tokens"],
                   "tokens_de_cache": respuesta.usage_metadata["input_token_details"].get("cache_read", 0),
                   "usd": usd})

## Escenario 4 · Claude: la caché se marca a mano

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

instrucciones = ("Eres un analista de datos. Escribe UNA consulta SQL (SQLite) sobre la "
                 "tabla sabana que responda la pregunta. Responde SOLO con el SQL.\n\n")

# En Claude se marca QUÉ parte cachear con cache_control. "ephemeral" = se
# guarda 5 minutos. Todo lo que está antes de la marca queda en caché.
sistema_cacheado = SystemMessage(content=[{
    "type": "text",
    "text": instrucciones + contexto,
    "cache_control": {"type": "ephemeral"},     # <- la marca
}])

# Primera llamada: ESCRIBE en la caché (cuesta un poco más que la entrada normal).
respuesta = claude.invoke([sistema_cacheado, HumanMessage(pregunta)])
print("SQL:", respuesta.text.strip().removeprefix("```sql").removesuffix("```").strip())
usd = costo(respuesta, "Claude")
escenarios.append({"escenario": "4a. Claude, escribe la caché", "proveedor": "Claude",
                   "tokens_entrada": respuesta.usage_metadata["input_tokens"],
                   "tokens_de_cache": respuesta.usage_metadata["input_token_details"].get("cache_read", 0),
                   "usd": usd})
print()

# Segunda llamada, otra pregunta: LEE de la caché (cuesta la décima parte).
respuesta = claude.invoke([sistema_cacheado, HumanMessage(otra_pregunta)])
print("SQL:", respuesta.text.strip().removeprefix("```sql").removesuffix("```").strip())
usd = costo(respuesta, "Claude")
escenarios.append({"escenario": "4b. Claude, lee la caché", "proveedor": "Claude",
                   "tokens_entrada": respuesta.usage_metadata["input_tokens"],
                   "tokens_de_cache": respuesta.usage_metadata["input_token_details"].get("cache_read", 0),
                   "usd": usd})

## La comparación

In [ ]:
comparacion = pd.DataFrame(escenarios)
comparacion["usd_por_1000_preguntas"] = comparacion["usd"] * 1000
comparacion.style.format({"usd": "{:.6f}", "usd_por_1000_preguntas": "{:.2f}",
                          "tokens_entrada": "{:,}", "tokens_de_cache": "{:,}"})

---

# Parte 4 · La tool: el LLM escribe, nuestro código ejecuta

Hasta ahora corríamos el SQL nosotros, a mano. Una **tool** (herramienta) es una
función de Python que el modelo puede **pedir** que se ejecute. Así el modelo
decide cuándo consultar, y nuestro código decide si lo cumple.

## Paso 1 · Definir la tool

In [ ]:
import json
from langchain_core.tools import tool

@tool   # este decorador convierte la función en una tool que el modelo puede pedir
def ejecutar_sql(sql: str) -> str:
    """Ejecuta una consulta SQL de solo lectura sobre la tabla sabana (SQLite) y devuelve las filas."""
    # El docstring de arriba NO es un comentario más: es lo que el modelo lee
    # para decidir cuándo usar la tool.
    conexion = sqlite3.connect(BASE.as_uri() + "?mode=ro", uri=True)   # ro = solo lectura
    filas = conexion.execute(sql).fetchall()
    conexion.close()
    return json.dumps(filas, ensure_ascii=False)    # las tools devuelven texto

## Paso 2 · La ficha que ve el modelo

In [ ]:
# LangChain arma la ficha sola, a partir del nombre, el docstring y los tipos.
print("Nombre     :", ejecutar_sql.name)
print("Descripción:", ejecutar_sql.description)
print("Parámetros :", ejecutar_sql.args)

## Paso 3 · Ejecutarla a mano

In [ ]:
# Una tool se puede probar sola, sin modelo: es una función como cualquier otra.
print(ejecutar_sql.invoke({"sql": "SELECT count(*) FROM sabana"}))

## Paso 4 · Entregársela al modelo

In [ ]:
# bind_tools le dice al modelo qué tools existen (le pasa sus fichas).
# parallel_tool_calls=False: una consulta por vuelta, para poder auditar cada SQL.
modelo_con_tool = foundry.bind_tools([ejecutar_sql], parallel_tool_calls=False)

sistema = ("Eres un analista de siniestralidad. Usa la tool ejecutar_sql para consultar "
           "la tabla sabana. Declara qué definición de costo usaste.\n\n" + contexto)

mensajes = [SystemMessage(sistema), HumanMessage(pregunta)]
pedido = modelo_con_tool.invoke(mensajes)

# El modelo no responde con texto: responde con un PEDIDO de tool.
# tool_calls es una lista (puede pedir varias); cada pedido trae nombre, argumentos e id.
print("Texto :", repr(pedido.text))
print("Tool  :", pedido.tool_calls[0]["name"])
print("SQL   :\n")
print(pedido.tool_calls[0]["args"]["sql"])
usd = costo(pedido, "Foundry")

## Paso 5 · Nuestro código ejecuta

In [ ]:
# Le pasamos el pedido completo a la tool. Devuelve un ToolMessage: el resultado,
# ya marcado con el id del pedido al que responde.
resultado = ejecutar_sql.invoke(pedido.tool_calls[0])

print("Tipo     :", type(resultado).__name__)
print("Resultado:", resultado.content)

## Paso 6 · El modelo redacta la respuesta

In [ ]:
import textwrap

mensajes.append(pedido)      # su propio pedido
mensajes.append(resultado)   # el resultado de la tool

final = modelo_con_tool.invoke(mensajes)

print("Agente:\n")
# textwrap.fill corta cada línea a 90 caracteres, respetando los saltos de línea del modelo.
print("\n".join(textwrap.fill(linea, 90) for linea in final.text.splitlines()))
print(f"\nRespuesta correcta: {real:,.2f}")
usd = costo(final, "Foundry")

---

# Parte 5 · La pregunta que rompe la cadena

Lo que acabamos de armar es una **cadena**: un camino fijo de pasos. Pregunta →
el modelo pide la tool **una vez** → se ejecuta → el modelo redacta.

Ahora, la pregunta que haría cualquier gerente después de ver la cifra.

In [ ]:
seguimiento = "¿Y comparado con el año anterior, en qué grupos de enfermedad se concentró el alza?"

# EXACTAMENTE los mismos pasos 4, 5 y 6, con la pregunta nueva.
mensajes = [SystemMessage(sistema), HumanMessage(seguimiento)]

pedido = modelo_con_tool.invoke(mensajes)                  # paso 4: pide la tool
usd = costo(pedido, "Foundry")
print("\nPrimer SQL que pidió:\n")
print(pedido.tool_calls[0]["args"]["sql"], "\n")

resultado = ejecutar_sql.invoke(pedido.tool_calls[0])      # paso 5: se ejecuta
mensajes += [pedido, resultado]

final = modelo_con_tool.invoke(mensajes)                   # paso 6: debería redactar
usd = costo(final, "Foundry")

print("\nRespuesta del agente:\n")
print("\n".join(textwrap.fill(linea, 90) for linea in final.text.splitlines()) or "(vacía)")
if final.tool_calls:
    print("\nEn lugar de responder, el modelo pidió OTRA consulta:\n")
    print(final.tool_calls[0]["args"]["sql"])

## Verifiquemos contra la base

In [ ]:
# Lo que de verdad pasó, calculado con SQL y no de memoria.
# CASE WHEN pone los dos años lado a lado en la misma fila.
hospitalario = con.execute("""
    SELECT sum(CASE WHEN anio = 2008 THEN monto_pagado END),
           sum(CASE WHEN anio = 2009 THEN monto_pagado END)
    FROM sabana WHERE ambito = 'Hospitalario'
""").fetchone()
print(f"Hospitalario 2008: {hospitalario[0]:>15,.2f}")
print(f"Hospitalario 2009: {hospitalario[1]:>15,.2f}   ({hospitalario[1] / hospitalario[0] - 1:+.1%})")
print("-> En hospitalización NO hubo alza: bajó.\n")

print("Mayores alzas por grupo, ambos ámbitos (millones):")
for grupo, antes, despues in con.execute("""
    SELECT grupo_enfermedad,
           sum(CASE WHEN anio = 2008 THEN monto_pagado END) / 1e6 AS antes,
           sum(CASE WHEN anio = 2009 THEN monto_pagado END) / 1e6 AS despues
    FROM sabana GROUP BY grupo_enfermedad
    ORDER BY despues - antes DESC LIMIT 4
"""):
    print(f"  {grupo[:50]:<50} {antes:6.2f} -> {despues:6.2f}  ({despues - antes:+.2f})")

### Por qué falló

Compara la respuesta del agente con la verificación. Según la corrida, la cadena
falla de una de estas formas, y todas vienen de lo mismo:

- **No recuerda la pregunta anterior.** Cada llamada empieza de cero: ya no sabe
  que hablábamos de hospitalización en 2009, y responde sobre otra cosa.
- **Una sola vuelta.** Si para comparar necesita una segunda consulta, la pide…
  y la cadena ya terminó: la respuesta queda vacía.
- **Hace las cuentas de memoria.** Cuando responde, resta y ordena en prosa, y
  se puede equivocar (en una corrida dijo +4,21 millones donde eran +4,51).
- **Nadie revisa la premisa.** El costo hospitalario **bajó** 4,9 %. "¿Dónde se
  concentró el alza?" no tiene respuesta en hospitalización: el alza está en
  ambulatorio.

Para eso hace falta un agente que **guarde lo que va sabiendo**, que pueda
**volver a consultar** y que **valide** antes de responder. Eso es LangGraph:
notebook 03.

## Lo que gastamos en este notebook

In [ ]:
con.close()
print(f"Gasto total de este notebook: USD {gasto_total:.6f}")
print(f"Correrlo 100 veces costaría:  USD {gasto_total * 100:.4f}")

---

## Cierre

- El LLM **escribe** el SQL; nuestro código lo **ejecuta**. La tool es solo una
  función con un docstring que el modelo lee.
- Sin diccionario, el SQL corre pero responde mal. **El agente no falla por el
  modelo: falla por no saber qué significa cada columna.**
- El diccionario cuesta tokens en cada pregunta; la **caché** lo abarata.

### Decisión de arquitectura · ¿diccionario entero en el prompt o búsqueda?

Aquí el diccionario mide unos 3 600 tokens y va **entero, en caché**. Con
cientos de tablas no cabría: ahí se indexa y se trae solo lo pertinente a cada
pregunta (búsqueda vectorial). **Mientras quepa, entero y en caché es más
simple, más barato y no se equivoca al buscar.**

### Tres preguntas de entrevista

1. ¿Por qué un SQL que corre sin error puede ser una respuesta equivocada? ¿Qué
   pondrías en el contexto del modelo para evitarlo?
2. La caché de prompts exige que el inicio del prompt sea idéntico. ¿Qué cosas
   la rompen sin que te des cuenta?
3. En el flujo de la tool, ¿en qué punto exacto pondrías un control antes de
   ejecutar el SQL que escribió el modelo, y qué revisaría?